# 이전 문맥 1~5문장 통제 실험

Top50의 문장 분리와 claim span을 고정하고, HCX에 제공하는 이전 문장 수만 1~5로 바꿉니다. 각 방법은 `in_ready=Y/N` 전체 파일을 보존하며 중단 후 같은 실행 셀을 다시 돌리면 이어받습니다.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/rnwjdgus03/NLP_05-Team-Project-3.git'
BRANCH = 'codex/repro-baseline-20260727'
REPO_DIR = Path('/content/NLP_05-Team-Project-3')
DRIVE_ROOT = Path('/content/drive/MyDrive/NLP_05-Team-Project-3')
BASE_RUN = DRIVE_ROOT / 'runs' / 'contextual_top50_context_v2_8x3'
RUN_DIR = DRIVE_ROOT / 'runs' / 'context_window_ablation_v1'
INDEX_DIR = DRIVE_ROOT / 'indexes' / 'kosis_bge_m3'
SENTENCES = BASE_RUN / '01_sentences.csv'
SPANS = BASE_RUN / '03_claim_spans.csv'
META_CANDIDATES = [
    DRIVE_ROOT / 'runs' / 'early_bge_rag_5000' / 'early_bge_meta_index.csv',
    DRIVE_ROOT / 'runs' / 'early_bge_rag' / 'early_bge_meta_index.csv',
]
META_INDEX = next((path for path in META_CANDIDATES if path.exists()), None)
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('run:', RUN_DIR)
print('meta:', META_INDEX)

In [ ]:
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'requests>=2.31,<3', 'python-dotenv>=1.0,<2',
    'numpy>=1.26,<3', 'sentence-transformers>=3.4,<6',
    'transformers>=4.45,<6'
], check=True)
os.chdir(REPO_DIR)
print('commit:', subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], text=True).strip())

In [ ]:
for path, label in [(SENTENCES, '고정 sentence CSV'), (SPANS, '고정 claim span CSV'), (INDEX_DIR / 'manifest.json', 'BGE-M3 index')]:
    if not path.exists():
        raise FileNotFoundError(f'{label}가 없습니다: {path}')
if not os.environ.get('CLOVA_API_KEY'):
    os.environ['CLOVA_API_KEY'] = userdata.get('CLOVA_API_KEY') or ''
if not os.environ.get('CLOVA_API_KEY'):
    raise RuntimeError('Colab 보안 비밀에 CLOVA_API_KEY를 등록하세요.')
print('inputs and API key: ready')

## 1~5문장 실행

이 셀은 오래 걸립니다. 중단되면 그대로 다시 실행하세요. 기존 checkpoint를 읽고 완료되지 않은 claim부터 이어서 처리합니다.

In [ ]:
command = [
    sys.executable, '-u', str(REPO_DIR / 'run_context_window_ablation.py'),
    '--sentences', str(SENTENCES),
    '--spans', str(SPANS),
    '--semantic-index', str(INDEX_DIR),
    '--out-dir', str(RUN_DIR),
    '--windows', '1', '2', '3', '4', '5',
    '--next-window', '0',
    '--related-limit', '0',
    '--lead-sentences', '3',
    '--device', 'cuda',
]
if META_INDEX:
    command.extend(['--meta-index', str(META_INDEX)])
print(' '.join(command))
subprocess.run(command, check=True)

## 공통 골드 평가

F1을 우선 비교합니다. `unseenY`가 0보다 크면 새 READY 후보를 공통 골드에 추가 판정한 뒤 최종 수치를 확정합니다.

In [ ]:
evaluation_dir = RUN_DIR / 'evaluation'
gold = REPO_DIR / 'data' / 'gold' / 'context_top50_common_gold_v1.csv'
eval_command = [
    sys.executable, '-u', str(REPO_DIR / 'evaluate_context_window_ablation.py'),
    '--gold', str(gold),
    '--out-dir', str(evaluation_dir),
]
for window in range(1, 6):
    eval_command.extend([
        '--run',
        f'prev_{window}={RUN_DIR / f"prev_{window}" / "06_in_ready_all.csv"}',
    ])
subprocess.run(eval_command, check=True)
print('metrics:', evaluation_dir / 'context_window_metrics.csv')
print('new READY review:', evaluation_dir / 'context_window_unseen_ready.csv')